#### Imports

In [ ]:
# Make utils/ importable when running this notebook from notebooks/
import sys
from pathlib import Path

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [ ]:
# Third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import iqr
from sklearn.metrics import r2_score, root_mean_squared_error

# Shared utils
from utils.checkpoint import Checkpoint
from utils.data_loading import load_feature_list, load_run_config
from utils.plotting import plot_error_histogram, set_theme
from utils.stats import REGION_CODE_MAP, STATE_CODE_MAP, grouped_error_stats

## 1. Setup

In [ ]:
cp = Checkpoint("Start")

#### Paths and files

In [ ]:
# Testing needs real census ground truth
config = load_run_config("D:/.../eval_config.yaml")

year = config["year"]
cell_col = config["cell_col"]
ags_col = config["ags_col"]
census_csv = config["census_csv"]
features_path = config["features"]

eval_dir = Path(config["eval_dir"])
eval_dir.mkdir(parents=True, exist_ok=True)
output_dir = Path(config["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
input_csv = output_dir / f"population_estimates_cells_{year}.csv"

## 2. Data loading

In [ ]:
input_data = pd.read_csv(input_csv, sep=";", converters={cell_col: str, "REG_TYPE": str})

reference_data = pd.read_csv(census_csv, sep=";",
                             converters={"BL": str, "CellCode": str, "AGS_2022": str})
reference_data[ags_col] = reference_data["AGS_2022"].str.zfill(8)

In [ ]:
independent_variables = len(load_feature_list(features_path)["features"])
print(f"Independent variables: {independent_variables}")

In [ ]:
cp.mark("Data loaded");

## 3. Statistics calculation

In [ ]:
def build_statistics_table(
    input_data, 
    reference_data, 
    cell_col
):
    merged = reference_data.merge(input_data, left_on="CellCode", right_on=cell_col, how="outer")

    merged.loc[merged["Einwohner"].isna() & merged["POPULATION_EST"].notna(), "Einwohner"] = 0
    merged.loc[merged["POPULATION_EST"].isna() & merged["Einwohner"].notna(), "POPULATION_EST"] = 0

    table = merged.loc[merged["Einwohner"].notna() & merged["POPULATION_EST"].notna()].reset_index(drop=True)
    table["OCCUPATION_STATUS"] = np.where(table["Einwohner"] == 0, "Unoccupied", "Occupied")
    return table


def add_error_columns(table):
    actuals = table["Einwohner"]
    predictions = table["POPULATION_EST"]
    error = predictions - actuals

    table["ERROR"] = error
    table["ABS_ERROR"] = abs(error)
    table["REL_ERROR"] = (error / actuals) * 100
    table["ABS_PERC_ERROR"] = (abs(error) / abs(actuals)) * 100

    # Both zero is a perfect prediction, not an undefined ratio
    table.loc[(actuals == 0) & (predictions == 0), ["REL_ERROR", "ABS_PERC_ERROR"]] = 0
    table.loc[table["REL_ERROR"].isin([np.inf, -np.inf]), ["REL_ERROR", "ABS_PERC_ERROR"]] = np.nan

    # IHSE (Inverse Hyperbolic Sine Error)
    table["IHSE"] = np.arcsinh(predictions) - np.arcsinh(actuals)
    table["ABS_IHSE"] = abs(table["IHSE"])
    table["FOLD_ERROR"] = np.exp(table["ABS_IHSE"])
    return table


def assign_quality_classes(table):
    """IHSE-based quality: A within ~1.1x, B ~1.5x, C ~2x, D ~3x, F worse."""
    bins = [-np.inf, np.log(1.1), np.log(1.5), np.log(2.0), np.log(3.0), np.inf]
    labels = ["A", "B", "C", "D", "F"]
    table["QUALI"] = pd.cut(table["ABS_IHSE"], bins=bins, labels=labels, right=True).astype(str)
    return table

In [ ]:
statistics_table = build_statistics_table(input_data, reference_data, cell_col)
statistics_table = add_error_columns(statistics_table)
statistics_table = assign_quality_classes(statistics_table)
statistics_table

In [ ]:
cp.mark("Statistics table built");

In [ ]:
actuals = statistics_table["Einwohner"]
predictions = statistics_table["POPULATION_EST"]

abs_error_percentiles = statistics_table["ABS_ERROR"].quantile([0.25, 0.5, 0.75])
abs_perc_error_percentiles = statistics_table["ABS_PERC_ERROR"].quantile([0.25, 0.5, 0.75])

print(f"ABS_ERROR -- 25th: {abs_error_percentiles[0.25]}, "
      f"Median: {abs_error_percentiles[0.5]}, 75th: {abs_error_percentiles[0.75]}")
print(f"ABS_PERC_ERROR -- 25th: {abs_perc_error_percentiles[0.25]}, "
      f"Median: {abs_perc_error_percentiles[0.5]}, 75th: {abs_perc_error_percentiles[0.75]}")

ref_mean, ref_median, ref_max = actuals.mean(), actuals.median(), actuals.max()
print(f"Reference values -- Mean: {ref_mean}, Median: {ref_median}, Max: {ref_max}")

In [ ]:
# Quality-class summary
qpercentages = statistics_table["QUALI"].value_counts(normalize=True).sort_index() * 100
group_means = (statistics_table
               .groupby("QUALI", observed=True)[["ABS_ERROR", "ABS_PERC_ERROR", "ABS_IHSE"]]
               .mean().sort_index())

quality_table = pd.DataFrame({
    "Quality": qpercentages.index,
    "Percentage": qpercentages.values,
    "MAE": group_means["ABS_ERROR"].values,
    "MAPE": group_means["ABS_PERC_ERROR"].values,
    "MAIHSE": group_means["ABS_IHSE"].values,
    "MFE": np.exp(group_means["ABS_IHSE"].values),
})

quality_table.to_csv(eval_dir / f"test_quality_groups_{year}.csv", sep=";", index=False)
quality_table

In [ ]:
# Export the full per-cell table (analysis.ipynb reads this)
statistics_table.to_csv(eval_dir / f"test_predictions_cells_ihse_{year}.csv", sep=";", index=False)

## 4. Overall statistics

In [ ]:
r2 = r2_score(actuals, predictions)
n = len(statistics_table)
p = independent_variables
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
rmse = root_mean_squared_error(actuals, predictions)

statistics_export = pd.DataFrame({
    "Reference mean": ref_mean,
    "Reference median": ref_median,
    "Reference max": ref_max,
    "R-squared": r2,
    "Adjusted R-squared": adj_r2,
    "Root mean squared error (RMSE)": rmse,
    "Mean error (ME)": statistics_table["ERROR"].mean(),
    "Median error": statistics_table["ERROR"].median(),
    "Min error": statistics_table["ERROR"].min(),
    "Max error": statistics_table["ERROR"].max(),
    "Std error": statistics_table["ERROR"].std(),
    "Mean absolute error (MAE)": statistics_table["ABS_ERROR"].mean(),
    "Std absolute error": statistics_table["ABS_ERROR"].std(),
    "Absolute error 25th percentile": abs_error_percentiles[0.25],
    "Absolute error 50th percentile (Median)": abs_error_percentiles[0.5],
    "Absolute error 75th percentile": abs_error_percentiles[0.75],
    "Mean relative error (MRE)": statistics_table["REL_ERROR"].mean(),
    "Median relative error": statistics_table["REL_ERROR"].median(),
    "Mean absolute percentage error (MAPE)": statistics_table["ABS_PERC_ERROR"].mean(),
    "Absolute percentage error 25th percentile": abs_perc_error_percentiles[0.25],
    "Absolute percentage error 50th percentile (Median percentage)": abs_perc_error_percentiles[0.5],
    "Absolute percentage error 75th percentile": abs_perc_error_percentiles[0.75],
    "Mean inverse hyperbolic sine error": statistics_table["IHSE"].mean(),
    "Median inverse hyperbolic sine error": statistics_table["IHSE"].median(),
    "Std inverse hyperbolic sine error": statistics_table["IHSE"].std(),
    "Interquartile range (IQR) of inverse hyperbolic sine error": iqr(statistics_table["IHSE"]),
    "Mean absolute inverse hyperbolic sine error (MAIHSE)": statistics_table["ABS_IHSE"].mean(),
    "Root mean squared inverse hyperbolic sine error (RMSIHSE)": np.sqrt((statistics_table["IHSE"] ** 2).mean()),
    "Mean fold error (MFE)": statistics_table["FOLD_ERROR"].mean(),
}, index=[0])

statistics_export.to_csv(eval_dir / f"test_statistics_{year}.csv", sep=";", decimal=",", index=False)
statistics_export.T

In [ ]:
cp.mark("Overall statistics computed");

## 5. Plotting

In [ ]:
set_theme()

subsets = {
    "": ("Cell level", statistics_table),
    "_occupied": ("Only occupied cells",
                  statistics_table[statistics_table["OCCUPATION_STATUS"] == "Occupied"]),
    "_unoccupied": ("Only unoccupied cells",
                    statistics_table[statistics_table["OCCUPATION_STATUS"] == "Unoccupied"]),
}

#### Error

In [ ]:
for suffix, (desc, subset) in subsets.items():
    plot_error_histogram(
        subset["ERROR"],
        title=f"Histogram of Error \n ({desc}, excluding 1st/99th percentiles) \n [Test {year}] \n",
        xlabel="Error [inhabitants]",
        out_path=eval_dir / f"test_hist_error{suffix}_{year}.png",
    )

#### Absolute error

In [ ]:
for suffix, (desc, subset) in subsets.items():
    plot_error_histogram(
        subset["ABS_ERROR"],
        title=f"Histogram of Absolute Error \n ({desc}, excluding 99th percentile) \n [Test {year}] \n",
        xlabel="Absolute error [inhabitants]",
        out_path=eval_dir / f"test_hist_abserror{suffix}_{year}.png",
        percentiles=(None, 99),
    )

#### Relative error

In [ ]:
for suffix, (desc, subset) in subsets.items():
    plot_error_histogram(
        subset["REL_ERROR"],
        title=f"Histogram of Relative Error \n ({desc}, excluding 1st/99th percentiles) \n [Test {year}] \n",
        xlabel="Relative error [%]",
        out_path=eval_dir / f"test_hist_relerror{suffix}_{year}.png",
    )

In [ ]:
plot_error_histogram(
    statistics_table["ABS_PERC_ERROR"],
    title=f"Histogram of Absolute Percentage Error \n (Cell level, excluding 99th percentile) \n [Test {year}] \n",
    xlabel="Absolute percentage error [%]",
    out_path=eval_dir / f"test_hist_abspercerror_{year}.png",
    percentiles=(None, 99),
)

#### IHSE

In [ ]:
for suffix, (desc, subset) in subsets.items():
    plt.figure(figsize=(8, 6), dpi=120)
    plt.hist(subset["IHSE"], bins=500, edgecolor="0", zorder=2)
    plt.title(f"Histogram of IHSE \n (Inverse Hyperbolic Sine Error; {desc}) \n [Test {year}] \n",
              fontsize=12, color="0")
    plt.xlim(-np.log(3), np.log(3))
    plt.xlabel("IHSE", fontsize=12, color="0")
    plt.ylabel("Frequency", fontsize=12, color="0")
    plt.xticks(fontsize=10, color="0.3")
    plt.yticks(fontsize=10, color="0.3")
    plt.grid(axis="y", alpha=0.75, zorder=0)
    sns.despine()
    plt.savefig(eval_dir / f"test_hist_ihse_zoomed{suffix}_{year}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
cp.mark("Histograms plotted");

## 6. Grouped statistics

In [ ]:
# Group by federal state
bl_grouped_table = grouped_error_stats(statistics_table, "BL")
bl_grouped_table["BL"] = bl_grouped_table["BL"].map(STATE_CODE_MAP)
bl_grouped_table.to_csv(eval_dir / f"test_statistics_bundeslaender_{year}.csv",
                        sep=";", decimal=",", index=False)
bl_grouped_table

In [ ]:
# Group by region type
statistics_table["REGIOSTAR_17_GEN"] = statistics_table["REG_TYPE"].map(REGION_CODE_MAP)

rs_grouped_table = grouped_error_stats(statistics_table, "REG_TYPE",
                                       extra_agg={"REGIOSTAR_17_CAT": ("REGIOSTAR_17_GEN", "first")})
rs_grouped_table.to_csv(eval_dir / f"test_statistics_regiostar17_{year}.csv",
                        sep=";", decimal=",", index=False)
rs_grouped_table

In [ ]:
# Group by inhabitant class
inh_bins = [1, 10, 20, 50, 100, 500, float("inf")]
inh_labels = ["A (1-10)", "B (11-20)", "C (21-50)", "D (51-100)", "E (101-500)", "F (501+)"]

statistics_table["Group"] = pd.cut(statistics_table["Einwohner"], bins=inh_bins, labels=inh_labels,
                                   right=True, include_lowest=False)
statistics_table["Group"] = statistics_table["Group"].cat.add_categories("X (0)")
statistics_table.loc[statistics_table["Einwohner"] == 0, "Group"] = "X (0)"

q_grouped_table = grouped_error_stats(statistics_table, "Group",
                                      extra_agg={"n_cells": ("Einwohner", "size")},
                                      sort_by="MEAN_INHABITANTS")

# Move n_cells next to the group label
cols = q_grouped_table.columns.tolist()
cols.insert(1, cols.pop(cols.index("n_cells")))
q_grouped_table = q_grouped_table[cols]

q_grouped_table.to_csv(eval_dir / f"test_statistics_grouped_{year}.csv",
                       sep=";", decimal=",", index=False)
q_grouped_table

#### Script runtime

In [ ]:
cp.finish()